In [1]:
import uproot
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mplhep as hep

In [2]:
# Leer archivos de datos reales de electrones (Run2012B y Run2012C)
file_el_b = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/Run2012B_DoubleElectron.root")
tree_el_b = file_el_b['Events']

file_el_c = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/Run2012C_DoubleElectron.root")
tree_el_c = file_el_c['Events']

# Listar ramas y variables físicas de electrones
print(tree_el_b.keys())
print(["run", "luminosityBlock", "event",'nElectron', 'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_mass', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dz', 'Electron_dxyErr', 'Electron_dzErr'])
print(tree_el_c.keys())
print(["run", "luminosityBlock", "event",'nElectron', 'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_mass', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dz', 'Electron_dxyErr', 'Electron_dzErr'])

# Señal MC: H → ZZ* → 4e
file_higgs_4e = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/SMHiggsToZZTo4L.root")
tree_higgs_4e = file_higgs_4e['Events']

# Fondo MC: ZZ → 4e
file_zz_4e = uproot.open("C:/Users/HP/Documents/Practica Profesional/Analisis BosonHiggs/Data Sets/ZZTo4e.root")
tree_zz_4e = file_zz_4e['Events']

['run', 'luminosityBlock', 'event', 'PV_npvs', 'PV_x', 'PV_y', 'PV_z', 'nMuon', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass', 'Muon_charge', 'Muon_pfRelIso03_all', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dxyErr', 'Muon_dz', 'Muon_dzErr', 'nElectron', 'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_mass', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dxyErr', 'Electron_dz', 'Electron_dzErr', 'MET_pt', 'MET_phi']
['run', 'luminosityBlock', 'event', 'nElectron', 'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_mass', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dz', 'Electron_dxyErr', 'Electron_dzErr']
['run', 'luminosityBlock', 'event', 'PV_npvs', 'PV_x', 'PV_y', 'PV_z', 'nMuon', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass', 'Muon_charge', 'Muon_pfRelIso03_all', 'Muon_pfRelIso04_all', 'Muon_dxy', 'Muon_dxyErr', 'Muon_dz', 'Muon_dzErr', 'nElectron', 'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_mass', 'E

In [3]:
# Extraer ramas para electrones
ramas_el = ["run", "luminosityBlock", "event",'Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dz', 'Electron_dxyErr', 'Electron_dzErr']

# Datos reales de los periodos B y C
df_data_b_el = tree_el_b.arrays(ramas_el, library="ak")
df_data_c_el = tree_el_c.arrays(ramas_el, library="ak")

# Concatenar ambos periodos de datos reales en uno solo
df_data_el = ak.concatenate([df_data_b_el, df_data_c_el], axis=0)

# Simulaciones (MC)
df_higgs_el = tree_higgs_4e.arrays(ramas_el, library="ak")
df_zz_el = tree_zz_4e.arrays(ramas_el, library="ak")

print("Entradas en Datos (Run B + C):", len(df_data_el))
print("Entradas totales en Higgs:", tree_higgs_4e.num_entries)
print("Entradas totales en ZZ:", tree_zz_4e.num_entries)

Entradas en Datos (Run B + C): 54011828
Entradas totales en Higgs: 299973
Entradas totales en ZZ: 1499093


In [4]:
# Dividimos las listas en columnas individuales para 4 electrones

mask_el = ak.num(df_data_el["Electron_pt"]) >= 4
filtered_el = df_data_el[mask_el] 

# Creamos el DataFrame de Pandas extrayendo los valores directamente para electrones (Datos Reales)
df_4el = pd.DataFrame({ 
    "run": ak.to_numpy(filtered_el["run"]),
    "luminosityBlock": ak.to_numpy(filtered_el["luminosityBlock"]),
    "event": ak.to_numpy(filtered_el["event"]),
         # Electrón 1
        "pt1": ak.to_numpy(filtered_el["Electron_pt"][:, 0]),
        "eta1": ak.to_numpy(filtered_el["Electron_eta"][:, 0]),
        "phi1": ak.to_numpy(filtered_el["Electron_phi"][:, 0]),
        "charge1": ak.to_numpy(filtered_el["Electron_charge"][:, 0]),
        "iso1": ak.to_numpy(filtered_el["Electron_pfRelIso03_all"][:, 0]),
        "dxy1": ak.to_numpy(filtered_el["Electron_dxy"][:, 0]),
        "dz1": ak.to_numpy(filtered_el["Electron_dz"][:, 0]),
        "dxyErr1": ak.to_numpy(filtered_el["Electron_dxyErr"][:, 0]),
        "dzErr1": ak.to_numpy(filtered_el["Electron_dzErr"][:, 0]),
        # Electrón 2
        "pt2": ak.to_numpy(filtered_el["Electron_pt"][:, 1]),
        "eta2": ak.to_numpy(filtered_el["Electron_eta"][:, 1]),
        "phi2": ak.to_numpy(filtered_el["Electron_phi"][:, 1]),
        "charge2": ak.to_numpy(filtered_el["Electron_charge"][:, 1]),
        "iso2": ak.to_numpy(filtered_el["Electron_pfRelIso03_all"][:, 1]),
        "dxy2": ak.to_numpy(filtered_el["Electron_dxy"][:, 1]),
        "dz2": ak.to_numpy(filtered_el["Electron_dz"][:, 1]),
        "dxyErr2": ak.to_numpy(filtered_el["Electron_dxyErr"][:, 1]),
        "dzErr2": ak.to_numpy(filtered_el["Electron_dzErr"][:, 1]),
        # Electrón 3
        "pt3": ak.to_numpy(filtered_el["Electron_pt"][:, 2]),
        "eta3": ak.to_numpy(filtered_el["Electron_eta"][:, 2]),
        "phi3": ak.to_numpy(filtered_el["Electron_phi"][:, 2]),
        "charge3": ak.to_numpy(filtered_el["Electron_charge"][:, 2]),
        "iso3": ak.to_numpy(filtered_el["Electron_pfRelIso03_all"][:, 2]),
        "dxy3": ak.to_numpy(filtered_el["Electron_dxy"][:, 2]),
        "dz3": ak.to_numpy(filtered_el["Electron_dz"][:, 2]),
        "dxyErr3": ak.to_numpy(filtered_el["Electron_dxyErr"][:, 2]),
        "dzErr3": ak.to_numpy(filtered_el["Electron_dzErr"][:, 2]),
        # Electrón 4
        "pt4": ak.to_numpy(filtered_el["Electron_pt"][:, 3]),
        "eta4": ak.to_numpy(filtered_el["Electron_eta"][:, 3]),
        "phi4": ak.to_numpy(filtered_el["Electron_phi"][:, 3]),
        "charge4": ak.to_numpy(filtered_el["Electron_charge"][:, 3]),
        "iso4": ak.to_numpy(filtered_el["Electron_pfRelIso03_all"][:, 3]),
        "dxy4": ak.to_numpy(filtered_el["Electron_dxy"][:, 3]),
        "dz4": ak.to_numpy(filtered_el["Electron_dz"][:, 3]),
        "dxyErr4": ak.to_numpy(filtered_el["Electron_dxyErr"][:, 3]),
        "dzErr4": ak.to_numpy(filtered_el["Electron_dzErr"][:, 3]), 
    }
)
print("Eventos con 4 electrones extraídos:", len(df_4el))


# Para la señal de Higgs (canal de electrones)

mask_higgs_el = ak.num(df_higgs_el["Electron_pt"]) >= 4
filtered_higgs_el = df_higgs_el[mask_higgs_el]

df_4el_higgs = pd.DataFrame({  
    "run": ak.to_numpy(filtered_higgs_el["run"]),
    "luminosityBlock": ak.to_numpy(filtered_higgs_el["luminosityBlock"]),
    "event": ak.to_numpy(filtered_higgs_el["event"]),
    # Electrón 1
    "pt1": ak.to_numpy(filtered_higgs_el["Electron_pt"][:, 0]),
    "eta1": ak.to_numpy(filtered_higgs_el["Electron_eta"][:, 0]),
    "phi1": ak.to_numpy(filtered_higgs_el["Electron_phi"][:, 0]),
    "charge1": ak.to_numpy(filtered_higgs_el["Electron_charge"][:, 0]),
    "iso1": ak.to_numpy(filtered_higgs_el["Electron_pfRelIso03_all"][:, 0]),
    "dxy1": ak.to_numpy(filtered_higgs_el["Electron_dxy"][:, 0]),
    "dz1": ak.to_numpy(filtered_higgs_el["Electron_dz"][:, 0]),
    "dxyErr1": ak.to_numpy(filtered_higgs_el["Electron_dxyErr"][:, 0]),
    "dzErr1": ak.to_numpy(filtered_higgs_el["Electron_dzErr"][:, 0]),
    # Electrón 2
    "pt2": ak.to_numpy(filtered_higgs_el["Electron_pt"][:, 1]),
    "eta2": ak.to_numpy(filtered_higgs_el["Electron_eta"][:, 1]),
    "phi2": ak.to_numpy(filtered_higgs_el["Electron_phi"][:, 1]),
    "charge2": ak.to_numpy(filtered_higgs_el["Electron_charge"][:, 1]),
    "iso2": ak.to_numpy(filtered_higgs_el["Electron_pfRelIso03_all"][:, 1]),
    "dxy2": ak.to_numpy(filtered_higgs_el["Electron_dxy"][:, 1]),
    "dz2": ak.to_numpy(filtered_higgs_el["Electron_dz"][:, 1]),
    "dxyErr2": ak.to_numpy(filtered_higgs_el["Electron_dxyErr"][:, 1]),
    "dzErr2": ak.to_numpy(filtered_higgs_el["Electron_dzErr"][:, 1]),
    # Electrón 3
    "pt3": ak.to_numpy(filtered_higgs_el["Electron_pt"][:, 2]),
    "eta3": ak.to_numpy(filtered_higgs_el["Electron_eta"][:, 2]),
    "phi3": ak.to_numpy(filtered_higgs_el["Electron_phi"][:, 2]),
    "charge3": ak.to_numpy(filtered_higgs_el["Electron_charge"][:, 2]),
    "iso3": ak.to_numpy(filtered_higgs_el["Electron_pfRelIso03_all"][:, 2]),
    "dxy3": ak.to_numpy(filtered_higgs_el["Electron_dxy"][:, 2]),
    "dz3": ak.to_numpy(filtered_higgs_el["Electron_dz"][:, 2]),
    "dxyErr3": ak.to_numpy(filtered_higgs_el["Electron_dxyErr"][:, 2]),
    "dzErr3": ak.to_numpy(filtered_higgs_el["Electron_dzErr"][:, 2]),
    # Electrón 4
    "pt4": ak.to_numpy(filtered_higgs_el["Electron_pt"][:, 3]),
    "eta4": ak.to_numpy(filtered_higgs_el["Electron_eta"][:, 3]),
    "phi4": ak.to_numpy(filtered_higgs_el["Electron_phi"][:, 3]),
    "charge4": ak.to_numpy(filtered_higgs_el["Electron_charge"][:, 3]),
    "iso4": ak.to_numpy(filtered_higgs_el["Electron_pfRelIso03_all"][:, 3]),
    "dxy4": ak.to_numpy(filtered_higgs_el["Electron_dxy"][:, 3]),
    "dz4": ak.to_numpy(filtered_higgs_el["Electron_dz"][:, 3]),
    "dxyErr4": ak.to_numpy(filtered_higgs_el["Electron_dxyErr"][:, 3]),
    "dzErr4": ak.to_numpy(filtered_higgs_el["Electron_dzErr"][:, 3]), 
})
print("Eventos de Higgs con 4 electrones extraídos:", len(df_4el_higgs))



# Mostrar las nuevas columnas de datos reales (Electrones)
print("Datos Reales (Electrones)")
print(df_4el[["pt1","pt2","pt3","pt4","eta1","eta2","eta3","eta4","phi1","phi2","phi3","phi4","charge1","charge2","charge3","charge4","iso1","iso2","iso3","iso4","dxyErr1","dzErr1"]].head())

# Mostrar las nuevas columnas de la señal de Higgs (Electrones)
print("\nSeñal Higgs (Electrones)")
print(df_4el_higgs[["pt1","pt2","pt3","pt4","eta1","eta2","eta3","eta4","phi1","phi2","phi3","phi4","charge1","charge2","charge3","charge4","iso1","iso2","iso3","iso4","dxyErr1","dzErr1"]].head())

# Valores únicos de las cargas (Datos Reales - Electrones)
print("Datos Reales (Electrones)")
print("Valores únicos en charge1:", df_4el["charge1"].unique())
print("Valores únicos en charge2:", df_4el["charge2"].unique())
print("Valores únicos en charge3:", df_4el["charge3"].unique())
print("Valores únicos en charge4:", df_4el["charge4"].unique())
print("Eventos totales con 4 electrones listos para procesar (Datos):", len(df_4el))

# Valores únicos de las cargas (Higgs Monte Carlo - Electrones)
print("\nSeñal Higgs (Electrones)")
print("Valores únicos en charge1:", df_4el_higgs["charge1"].unique())
print("Valores únicos en charge2:", df_4el_higgs["charge2"].unique())
print("Valores únicos en charge3:", df_4el_higgs["charge3"].unique())
print("Valores únicos en charge4:", df_4el_higgs["charge4"].unique())
print("Eventos totales con 4 electrones listos para procesar (Higgs):", len(df_4el_higgs))

Eventos con 4 electrones extraídos: 652071
Eventos de Higgs con 4 electrones extraídos: 21969
Datos Reales (Electrones)
         pt1        pt2        pt3        pt4      eta1      eta2      eta3  \
0  13.930800  20.208078   6.319327  13.284051  1.272109  1.330733  1.906258   
1   8.294188  13.615106   5.693320  24.751520 -2.134267  0.617290 -1.962833   
2  11.130920   8.427904  11.312430  38.044636  0.012020 -0.985687 -0.112611   
3   6.910966  16.960032  21.457500  88.581207 -1.450213  1.728875 -0.130935   
4   7.798933   9.856538  36.629272  31.462450 -0.339343 -0.847481 -1.349068   

       eta4      phi1      phi2  ...  charge1  charge2  charge3  charge4  \
0  1.083326  2.636612  2.658260  ...        1       -1       -1        1   
1 -2.196459 -1.553178 -0.804566  ...       -1       -1       -1        1   
2  2.324481  1.633967 -1.893318  ...        1        1        1        1   
3 -2.046107 -2.009760  1.912342  ...        1        1       -1       -1   
4  1.281632  0.520511 -2.

In [5]:
# Para el fondo ZZ → 4e (canal de electrones)

mask_zz_el = ak.num(df_zz_el["Electron_pt"]) >= 4
filtered_zz_el = df_zz_el[mask_zz_el]

df_4el_zz = pd.DataFrame({ 
    "run": ak.to_numpy(filtered_zz_el["run"]),
    "luminosityBlock": ak.to_numpy(filtered_zz_el["luminosityBlock"]),
    "event": ak.to_numpy(filtered_zz_el["event"]), 
    # Electrón 1
    "pt1": ak.to_numpy(filtered_zz_el["Electron_pt"][:, 0]),
    "eta1": ak.to_numpy(filtered_zz_el["Electron_eta"][:, 0]),
    "phi1": ak.to_numpy(filtered_zz_el["Electron_phi"][:, 0]),
    "charge1": ak.to_numpy(filtered_zz_el["Electron_charge"][:, 0]),
    "iso1": ak.to_numpy(filtered_zz_el["Electron_pfRelIso03_all"][:, 0]),
    "dxy1": ak.to_numpy(filtered_zz_el["Electron_dxy"][:, 0]),
    "dz1": ak.to_numpy(filtered_zz_el["Electron_dz"][:, 0]),
    "dxyErr1": ak.to_numpy(filtered_zz_el["Electron_dxyErr"][:, 0]),
    "dzErr1": ak.to_numpy(filtered_zz_el["Electron_dzErr"][:, 0]),
    # Electrón 2
    "pt2": ak.to_numpy(filtered_zz_el["Electron_pt"][:, 1]),
    "eta2": ak.to_numpy(filtered_zz_el["Electron_eta"][:, 1]),
    "phi2": ak.to_numpy(filtered_zz_el["Electron_phi"][:, 1]),
    "charge2": ak.to_numpy(filtered_zz_el["Electron_charge"][:, 1]),
    "iso2": ak.to_numpy(filtered_zz_el["Electron_pfRelIso03_all"][:, 1]),
    "dxy2": ak.to_numpy(filtered_zz_el["Electron_dxy"][:, 1]),
    "dz2": ak.to_numpy(filtered_zz_el["Electron_dz"][:, 1]),
    "dxyErr2": ak.to_numpy(filtered_zz_el["Electron_dxyErr"][:, 1]),
    "dzErr2": ak.to_numpy(filtered_zz_el["Electron_dzErr"][:, 1]),
    # Electrón 3
    "pt3": ak.to_numpy(filtered_zz_el["Electron_pt"][:, 2]),
    "eta3": ak.to_numpy(filtered_zz_el["Electron_eta"][:, 2]),
    "phi3": ak.to_numpy(filtered_zz_el["Electron_phi"][:, 2]),
    "charge3": ak.to_numpy(filtered_zz_el["Electron_charge"][:, 2]),
    "iso3": ak.to_numpy(filtered_zz_el["Electron_pfRelIso03_all"][:, 2]),
    "dxy3": ak.to_numpy(filtered_zz_el["Electron_dxy"][:, 2]),
    "dz3": ak.to_numpy(filtered_zz_el["Electron_dz"][:, 2]),
    "dxyErr3": ak.to_numpy(filtered_zz_el["Electron_dxyErr"][:, 2]),
    "dzErr3": ak.to_numpy(filtered_zz_el["Electron_dzErr"][:, 2]),
    # Electrón 4
    "pt4": ak.to_numpy(filtered_zz_el["Electron_pt"][:, 3]),
    "eta4": ak.to_numpy(filtered_zz_el["Electron_eta"][:, 3]),
    "phi4": ak.to_numpy(filtered_zz_el["Electron_phi"][:, 3]),
    "charge4": ak.to_numpy(filtered_zz_el["Electron_charge"][:, 3]),
    "iso4": ak.to_numpy(filtered_zz_el["Electron_pfRelIso03_all"][:, 3]),
    "dxy4": ak.to_numpy(filtered_zz_el["Electron_dxy"][:, 3]),
    "dz4": ak.to_numpy(filtered_zz_el["Electron_dz"][:, 3]),
    "dxyErr4": ak.to_numpy(filtered_zz_el["Electron_dxyErr"][:, 3]),
    "dzErr4": ak.to_numpy(filtered_zz_el["Electron_dzErr"][:, 3]), 
})
print("Eventos con 4 electrones extraídos (Fondo ZZ):", len(df_4el_zz))

# Mostrar las nuevas columnas del fondo ZZ (Electrones)
print("\nFondo ZZ (Electrones)")
print(df_4el_zz[["pt1","pt2","pt3","pt4","eta1","eta2","eta3","eta4","phi1","phi2","phi3","phi4","charge1","charge2","charge3","charge4","iso1","iso2","iso3","iso4","dxyErr1","dzErr1"]].head())

# Valores únicos de las cargas (Fondo ZZ Monte Carlo - Electrones)
print("\nFondo ZZ (Electrones)")
print("Valores únicos en charge1:", df_4el_zz["charge1"].unique())
print("Valores únicos en charge2:", df_4el_zz["charge2"].unique())
print("Valores únicos en charge3:", df_4el_zz["charge3"].unique())
print("Valores únicos en charge4:", df_4el_zz["charge4"].unique())
print("Eventos totales con 4 electrones listos para procesar (ZZ):", len(df_4el_zz))

Eventos con 4 electrones extraídos (Fondo ZZ): 221004

Fondo ZZ (Electrones)
         pt1        pt2        pt3        pt4      eta1      eta2      eta3  \
0  18.510689  14.780485   7.352483  20.088581  0.847147  1.020884  0.682995   
1   9.049304  15.528327   9.776957  26.475550 -0.389181  0.267063 -0.697519   
2  11.022942  26.166174  41.077019  15.480583  2.225791  1.704516  1.578375   
3  28.738707  65.071678  78.531097  42.304951 -0.623607  0.075963 -1.202358   
4  14.476383  27.230831  29.415319   5.204156 -1.089972 -1.364240 -2.099011   

       eta4      phi1      phi2  ...  charge1  charge2  charge3  charge4  \
0 -1.223226 -1.639722 -1.285156  ...       -1        1        1       -1   
1 -2.443859  1.172508  0.626357  ...       -1       -1        1        1   
2  2.342604  1.121064  1.587350  ...       -1        1       -1        1   
3 -2.225029 -1.264189  2.070570  ...        1       -1        1       -1   
4 -2.219256 -3.046280  0.725744  ...       -1        1       -1     

In [6]:
# Guardar los DataFrames de electrones en formato Apache Parquet

df_4el.to_parquet("df_data_4el.parquet", index=False)
df_4el_higgs.to_parquet("df_higgs_4el.parquet", index=False)
df_4el_zz.to_parquet("df_zz_4el.parquet", index=False)

print("¡Archivos Parquet de electrones guardados exitosamente!")
print("- df_data_4el.parquet")
print("- df_higgs_4el.parquet")
print("- df_zz_4el.parquet")

¡Archivos Parquet de electrones guardados exitosamente!
- df_data_4el.parquet
- df_higgs_4el.parquet
- df_zz_4el.parquet
